Taught by Kash T, Aaditya P and Aryan M. Read this after the two week 15 lessons, [Data | SQL Connect](/2025/01/08/sqlconnect_IPYNB_2_.html) and [Data | SQLAlchemy](/2025/01/08/sqlalchemy_IPYNB_2_.html). This lesson connects them to a model from a real project.

## 1. Reference Guide (On Your Own)

| Term | Meaning | Example |
|---|---|---|
| **Table** | Rows of the same kind of data, all with the same columns | `labrats_inquiries` holds every LabRats website form submission |
| **Row** | One record in a table | One person's contact form |
| **Column** | One piece of data every row has, with a type | `email`, a string of up to 200 characters |
| **Primary key** | The column whose value is different on every row, so a row can be found by it | `id`. The database picks it when the row is created |
| **Model** | A Python class whose objects are rows of one table | `class LabRatsInquiry(db.Model)` |
| **Schema** | The list of tables, their columns and the rules on them | `form` is `NOT NULL`, `status` defaults to `'new'` |
| **Constraint** | A rule the database enforces, even if the Python code forgets | `NOT NULL` refuses a row with no email |
| **Foreign key** | A column that holds another table's primary key, linking the two | OCS `posts._user_id` points at `users.id` |
| **CRUD** | Create, Read, Update, Delete: the four things a program does with stored rows | `create()`, `read()`, `update_status()`, `delete()` |
| **Commit** | Saving the changes made since the last commit. Until then, other connections don't see them | `db.session.commit()` in SQLAlchemy, `db.commit()` in `sqlite3` |

---
## 2. Real-Life Application (In-Class)
### LabRats Website Forms

San Diego LabRats is our team's capstone. Its website has a contact form, a newsletter sign-up, a scholarship form and a few more. All of them are stored in **one** table through **one** model. Each form request goes through the same layers you met in Flask:

| What happens | Request | Code that runs | What the table sees |
|:---|:---|:---|:---|
| A parent sends the contact form | `POST /api/labrats/inquiries` | `validate_inquiry(data)`, then `LabRatsInquiry.from_cleaned(cleaned).create()` | `INSERT` one row with `status = 'new'` |
| An admin opens the inbox | `GET /api/labrats/inquiries?form=contact&status=new` | `LabRatsInquiry.query.filter_by(...).order_by(created_at.desc())` | `SELECT ... WHERE form = ? AND status = ? ORDER BY created_at DESC` |
| The admin marks it handled | `PUT /api/labrats/inquiries/7` with `{"status": "handled"}` | `inquiry.update_status("handled")` | `UPDATE ... SET status = 'handled' WHERE id = 7` |
| The request has a bad status | `PUT` with `{"status": "deleted"}` | the API checks `STATUSES` and answers `400` | nothing changes |

Two design choices worth noticing:

- **One table for six forms.** Columns every form has (`form`, `first_name`, `last_name`, `email`, `phone`, `status`, `created_at`) are real columns. Answers only one form has, like a scholarship's grade, go in a `details` column as JSON text. Six tables with mostly the same columns would mean six models to keep in step.
- **The API checks the data, and the database checks it again.** `validate_inquiry` gives the user a friendly message next to the field. `nullable=False` on the column refuses a bad row even if a future endpoint forgets to validate.

---
## 3. Tech Talk (In-Class)

**Learning Objective:** Design a model for your project's data, and write its CRUD methods so the object and the table always agree.

**Success Criteria:** You can name the table, columns, primary key and constraints of a model. You can say which SQL each CRUD method runs, and explain why changing an object without committing changes nothing in the table.

This is the LabRats model, shortened. The full file is `model/labrats_inquiry.py` in the [sdlabrats_backend](https://github.com/SDLabRats-Team/sdlabrats_backend) repo.

```python
class LabRatsInquiry(db.Model):
    __tablename__ = "labrats_inquiries"

    id = db.Column(db.Integer, primary_key=True)
    form = db.Column(db.String(32), nullable=False, index=True)
    first_name = db.Column(db.String(200), nullable=False)
    email = db.Column(db.String(200), nullable=False)
    details = db.Column(db.Text, nullable=False, default="{}")
    status = db.Column(db.String(16), nullable=False, default="new")
    created_at = db.Column(db.DateTime, default=datetime.utcnow, index=True)

    def create(self):
        db.session.add(self)
        db.session.commit()
        return self

    def update_status(self, status):
        if status not in STATUSES:
            raise ValueError(f"Unknown inquiry status '{status}'")
        self.status = status
        db.session.commit()
        return self
```

SQLAlchemy writes the SQL for you. Each line above stands for SQL you could write yourself:

| SQLAlchemy | SQL it runs |
|---|---|
| `__tablename__ = "labrats_inquiries"` | `CREATE TABLE labrats_inquiries (...)` |
| `db.Column(db.Integer, primary_key=True)` | `id INTEGER PRIMARY KEY` |
| `db.Column(db.String(200), nullable=False)` | `email VARCHAR(200) NOT NULL` |
| `default="new"` | `DEFAULT 'new'`, filled in when a row leaves it out |
| `db.session.add(obj)` then `commit()` | `INSERT INTO labrats_inquiries (...) VALUES (?, ?, ...)` |
| `Model.query.get(7)` | `SELECT * FROM labrats_inquiries WHERE id = 7` |
| `query.filter_by(form="contact")` | `... WHERE form = 'contact'` |
| changing `obj.status`, then `commit()` | `UPDATE labrats_inquiries SET status = ? WHERE id = ?` |
| `db.session.delete(obj)` then `commit()` | `DELETE FROM labrats_inquiries WHERE id = ?` |

The runners below use Python's built-in `sqlite3` module, so you can see that SQL. SQLite is also the database the OCS Flask backend uses on your laptop.

### Code Example (On Your Own)

#### A. A small Inquiry model with create and read

`CREATE TABLE` is the schema. The class is the model. `create()` runs an `INSERT`, and `read()` turns the object into a dictionary, the same shape an API sends back as JSON. The `?` marks are placeholders: `sqlite3` sends the values separately from the SQL text, so a name like `Robert'); DROP TABLE inquiries;--` is stored as text instead of running as SQL.

In [ ]:
# CODE_RUNNER: Run it, then create a third inquiry and predict the new row count before you run again
# A model is a Python class whose objects are rows in one table.
import sqlite3

db = sqlite3.connect(":memory:")  # a database that lives only while this program runs
db.execute("""
CREATE TABLE inquiries (
    id         INTEGER PRIMARY KEY,
    form       TEXT NOT NULL,
    first_name TEXT NOT NULL,
    email      TEXT NOT NULL,
    status     TEXT NOT NULL DEFAULT 'new'
)""")


class Inquiry:
    """One row of the inquiries table, as a Python object."""

    def __init__(self, form, first_name, email):
        self.id = None          # the database picks the id when the row is created
        self.form = form
        self.first_name = first_name
        self.email = email
        self.status = "new"

    def create(self):
        cursor = db.execute(
            "INSERT INTO inquiries (form, first_name, email) VALUES (?, ?, ?)",
            (self.form, self.first_name, self.email),
        )
        db.commit()
        self.id = cursor.lastrowid
        return self

    def read(self):
        return {"id": self.id, "form": self.form, "first_name": self.first_name,
                "email": self.email, "status": self.status}


ada = Inquiry("contact", "Ada", "ada@example.com").create()
lin = Inquiry("newsletter", "Lin", "lin@example.com").create()
print(ada.read())
print(lin.read())

# Try it: remove the # marks, then predict the row count before you run.
# sam = Inquiry("survey", "Sam", "sam@example.com").create()
# print(sam.read())

print("rows in the table:", db.execute("SELECT COUNT(*) FROM inquiries").fetchone()[0])

#### Popcorn Hack (In-Class)

2-minute challenge: predict before you run.

The runner below has a starter `update_status` that only does `self.status = status`. It creates one inquiry, calls `update_status("handled")`, then prints the object's status and the status stored in the table.

1. Write your prediction for **both** printed lines in the box below.
2. Run the starter and compare.
3. In one sentence, say what the starter forgot. Fixing it is Homework Hack 1.

<!-- REFLECTION_RUNNER: Before you run: what will "object says:" and "table says:" print, and why? | id: database-models-popcorn -->

In [ ]:
# CODE_RUNNER: Run the starter update_status and compare the two lines with your prediction
# The starter update_status has a bug. Do not fix it here: this runner only shows what it does.
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("""
CREATE TABLE inquiries (
    id     INTEGER PRIMARY KEY,
    form   TEXT NOT NULL,
    email  TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'new'
)""")


class Inquiry:
    def __init__(self, form, email):
        self.id = None
        self.form = form
        self.email = email
        self.status = "new"

    def create(self):
        cursor = db.execute("INSERT INTO inquiries (form, email) VALUES (?, ?)", (self.form, self.email))
        db.commit()
        self.id = cursor.lastrowid
        return self

    # Starter code: mark an inquiry as handled
    def update_status(self, status):
        self.status = status
        return self


ada = Inquiry("contact", "ada@example.com").create()
ada.update_status("handled")

print("object says:", ada.status)
row = db.execute("SELECT status FROM inquiries WHERE id = ?", (ada.id,)).fetchone()
print("table says: ", row[0])

## 4. MCQ Check (In-Class)

5 questions, one at a time. Answer, check, then go on to the next one.

<!-- MCQ_RUNNER: MCQ Database Models | id: database-models-mcq -->

```json
[
 {
  "q": "In a SQLAlchemy model, what becomes a column in the table?",
  "choices": [
   "Every method of the class",
   "Each class attribute set to db.Column(...)",
   "The class name",
   "Only the parameters of __init__"
  ],
  "answer": 1,
  "ref": "Tech Talk: SQLAlchemy and the SQL it runs",
  "why": "Each db.Column(...) attribute becomes one column. Methods like create() are behavior, not stored data, and __tablename__ names the table."
 },
 {
  "q": "A LabRats admin marks a contact form as handled. Which CRUD operation and HTTP verb does that use?",
  "choices": [
   "Create, with POST",
   "Read, with GET",
   "Update, with PUT",
   "Delete, with DELETE"
  ],
  "answer": 2,
  "ref": "Real-Life Application: LabRats Website Forms",
  "why": "The row already exists and one column changes, so it is Update. The LabRats API sends it as PUT /api/labrats/inquiries/<id>."
 },
 {
  "q": "What does this print?",
  "code": "ada = Inquiry(\"contact\", \"ada@example.com\").create()\nada.status = \"handled\"   # no UPDATE, no commit\nrow = db.execute(\"SELECT status FROM inquiries WHERE id = ?\", (ada.id,)).fetchone()\nprint(row[0])",
  "choices": [
   "handled",
   "new",
   "None",
   "It raises an error"
  ],
  "answer": 1,
  "ref": "Popcorn Hack",
  "why": "Changing the Python object does not touch the table. Only an UPDATE (or a SQLAlchemy commit) saves the change, so the row still says new."
 },
 {
  "q": "Why does the code write \"WHERE id = ?\" and pass the id separately, instead of building the SQL with an f-string?",
  "choices": [
   "It runs faster",
   "The value is sent as data, so text typed by a user can't run as SQL",
   "SQLite can't read numbers inside SQL text",
   "The ? picks the next free id"
  ],
  "answer": 1,
  "ref": "Code Example A",
  "why": "With placeholders, sqlite3 sends the values separately from the SQL. A value can't end the statement and start a new one, which is how SQL injection works."
 },
 {
  "q": "Why is id the primary key of the inquiries table, instead of email?",
  "choices": [
   "Emails are too long to store",
   "The same person can send more than one form, so email is not different on every row",
   "Primary keys must be text",
   "SQLite needs exactly one integer column"
  ],
  "answer": 1,
  "ref": "Reference Guide: Primary key",
  "why": "A primary key must be different on every row. One parent can send a contact form and a scholarship form with the same email, so the table uses a number the database picks."
 }
]
```

---

## 5. Homework Hacks (On Your Own)

**Task:** Build up the `Inquiry` model one CRUD operation at a time. Each starter runs before you edit it, so you can run it first and read its `FAIL` lines. Change only what the TODO asks for, and keep going until the last line reads `---- N / N passed ----`.

Copy these lines into your submission notes as you finish each part:

```text
Lesson: CSP Database Models
Popcorn: <your prediction, and what the starter forgot>
MCQ Database Models: <your score out of 5>
Homework Hack 1: ---- <N> / 7 passed ----
Homework Hack 2: ---- <N> / 7 passed ----
Homework Hack 3: ---- <N> / 6 passed ----
```

Rules for every runner:

- Each runner is a complete program with its own `:memory:` database, so nothing you do is saved between runs.
- Don't edit anything below `tests: do not edit below this line`.
- Always pass values with `?` placeholders.

#### Hack 1: Update that saves (10 minutes)

Fix `update_status` so the object and the table agree. It must refuse a status that isn't in `STATUSES`, as the LabRats model does, and leave everything unchanged when it does.

In [ ]:
# CODE_RUNNER: Fix update_status until every check prints PASS
# update_status has a bug. Run the tests, read the FAIL lines, then fix it until you see 7 / 7 passed.
import sqlite3

STATUSES = ("new", "handled")

db = sqlite3.connect(":memory:")
db.execute("""
CREATE TABLE inquiries (
    id     INTEGER PRIMARY KEY,
    form   TEXT NOT NULL,
    email  TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'new'
)""")


class Inquiry:
    def __init__(self, form, email):
        self.id = None
        self.form = form
        self.email = email
        self.status = "new"

    def create(self):
        cursor = db.execute("INSERT INTO inquiries (form, email) VALUES (?, ?)", (self.form, self.email))
        db.commit()
        self.id = cursor.lastrowid
        return self

    # TODO: fix this method. A real update must:
    #   - raise ValueError if status is not in STATUSES (the API answers 400),
    #   - save the new status in the table with an UPDATE ... WHERE id = ?, then commit,
    #   - change self.status too, and return self.
    # A rejected status must leave both the object and the row exactly as they were.
    def update_status(self, status):
        self.status = status
        return self


# ---------- tests: do not edit below this line ----------
def stored_status(inquiry):
    return db.execute("SELECT status FROM inquiries WHERE id = ?", (inquiry.id,)).fetchone()[0]

passed = 0
total = 0
def check(name, condition):
    global passed, total
    total += 1
    if condition:
        passed += 1
    print(("PASS: " if condition else "FAIL: ") + name)

ada = Inquiry("contact", "ada@example.com").create()
lin = Inquiry("newsletter", "lin@example.com").create()
returned = ada.update_status("handled")
check("update_status returns the same object", returned is ada)
check("the object now says handled", ada.status == "handled")
check("the table row now says handled", stored_status(ada) == "handled")
check("the other inquiry is unchanged in the table", stored_status(lin) == "new")

try:
    lin.update_status("deleted")
    raised = False
except ValueError:
    raised = True
check("an unknown status raises ValueError", raised)
check("a rejected status leaves the object as it was", lin.status == "new")
check("a rejected status leaves the row as it was", stored_status(lin) == "new")
print(f"---- {passed} / {total} passed ----")

#### Hack 2: Read by id and by filter (10 minutes)

The LabRats inbox reads one inquiry by its id and lists the others with optional `form` and `status` filters, newest first. Write `find(inquiry_id)` and `all_by(form=None, status=None)`. Add a `WHERE` condition only for a filter that is given.

| Call | Expected |
|---|---|
| `find(3)` | the inquiry from `sam@example.com` |
| `find(99)` | `None` |
| `all_by()` | ids `[4, 3, 2, 1]` |
| `all_by(form="contact")` | ids `[3, 1]` |
| `all_by(form="contact", status="new")` | ids `[1]` |

In [ ]:
# CODE_RUNNER: Write find and all_by until every check prints PASS
# The admin page reads inquiries by id and by filters. Write both reads, then run until you see 7 / 7 passed.
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("""
CREATE TABLE inquiries (
    id     INTEGER PRIMARY KEY,
    form   TEXT NOT NULL,
    email  TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'new'
)""")
rows = [("contact", "ada@example.com", "new"), ("newsletter", "lin@example.com", "new"),
        ("contact", "sam@example.com", "handled"), ("survey", "kai@example.com", "new")]
db.executemany("INSERT INTO inquiries (form, email, status) VALUES (?, ?, ?)", rows)
db.commit()


class Inquiry:
    def __init__(self, form, email, status="new", id=None):
        self.id = id
        self.form = form
        self.email = email
        self.status = status

    @staticmethod
    def from_row(row):
        """Turn one SELECT result (id, form, email, status) into an Inquiry."""
        return Inquiry(form=row[1], email=row[2], status=row[3], id=row[0])

    # TODO: return the Inquiry with this id, or None if no row has it.
    # Use SELECT id, form, email, status FROM inquiries WHERE id = ?
    @staticmethod
    def find(inquiry_id):
        return None

    # TODO: return a list of Inquiry objects, newest first (highest id first).
    # form and status are optional filters: only add a WHERE condition for one that is given.
    # Always pass values with ? placeholders, never by gluing them into the SQL text.
    @staticmethod
    def all_by(form=None, status=None):
        return []


# ---------- tests: do not edit below this line ----------
passed = 0
total = 0
def check(name, condition):
    global passed, total
    total += 1
    if condition:
        passed += 1
    print(("PASS: " if condition else "FAIL: ") + name)

found = Inquiry.find(3)
check("find(3) returns an Inquiry", isinstance(found, Inquiry))
check("find(3) has the right email", found is not None and found.email == "sam@example.com")
check("find(99) returns None", Inquiry.find(99) is None)
everything = Inquiry.all_by()
check("all_by() returns all 4 inquiries", len(everything) == 4)
check("all_by() is newest first", [i.id for i in everything] == [4, 3, 2, 1])
check("all_by(form='contact') returns ids 3 and 1", [i.id for i in Inquiry.all_by(form="contact")] == [3, 1])
check("all_by(form='contact', status='new') returns only id 1",
      [i.id for i in Inquiry.all_by(form="contact", status="new")] == [1])
print(f"---- {passed} / {total} passed ----")

#### Hack 3: Create safely and delete (10 minutes)

The `email` column is `NOT NULL`, so the database refuses a row without one and raises `sqlite3.IntegrityError`. The OCS `Post` model handles this by rolling back and returning `None` instead of crashing. Do the same in `create()`, then write `delete()`.

In [ ]:
# CODE_RUNNER: Finish create and delete until every check prints PASS
# The table refuses an inquiry with no email. Make create handle that, write delete, and run until you see 6 / 6 passed.
import sqlite3

db = sqlite3.connect(":memory:")
db.execute("""
CREATE TABLE inquiries (
    id     INTEGER PRIMARY KEY,
    form   TEXT NOT NULL,
    email  TEXT NOT NULL,
    status TEXT NOT NULL DEFAULT 'new'
)""")


class Inquiry:
    def __init__(self, form, email):
        self.id = None
        self.form = form
        self.email = email
        self.status = "new"

    # TODO: the INSERT raises sqlite3.IntegrityError when a NOT NULL column gets None.
    # Catch it, call db.rollback(), and return None instead of crashing.
    def create(self):
        cursor = db.execute("INSERT INTO inquiries (form, email) VALUES (?, ?)", (self.form, self.email))
        db.commit()
        self.id = cursor.lastrowid
        return self

    # TODO: delete this inquiry's row (DELETE ... WHERE id = ?), commit,
    # and return True if a row was deleted or False if none was (cursor.rowcount tells you).
    def delete(self):
        return False


# ---------- tests: do not edit below this line ----------
def count():
    return db.execute("SELECT COUNT(*) FROM inquiries").fetchone()[0]

passed = 0
total = 0
def check(name, condition):
    global passed, total
    total += 1
    if condition:
        passed += 1
    print(("PASS: " if condition else "FAIL: ") + name)

ada = Inquiry("contact", "ada@example.com").create()
lin = Inquiry("newsletter", "lin@example.com").create()
try:
    blank = Inquiry("contact", None).create()
    crashed = False
except sqlite3.IntegrityError:
    blank = "crashed"
    crashed = True
check("create with no email does not crash", not crashed)
check("create with no email returns None", blank is None)
check("the table still has 2 rows", count() == 2)
check("delete returns True for a saved inquiry", ada.delete() is True)
check("the table now has 1 row", count() == 1)
check("deleting it again returns False", ada.delete() is False)
print(f"---- {passed} / {total} passed ----")

---

### Grading Plan (1 Point Total)

| Part | Points | What earns the points |
|---|---|---|
| Popcorn | 0.2 | A prediction for both lines written before running, and one sentence on what the starter forgot |
| MCQ | 0.2 | All 5 questions answered; the score is in your notes |
| Hack 1 | 0.2 | `---- 7 / 7 passed ----` |
| Hack 2 | 0.2 | `---- 7 / 7 passed ----` |
| Hack 3 | 0.2 | `---- 6 / 6 passed ----` |

---

## 6. Apply It to Your Project

Pick one table your project's backend stores, or will store. Write down:

1. its table name and model class name
2. each column with its type, and which ones are `nullable=False`
3. its primary key, and any foreign key to another table
4. which API request runs each of create, read, update and delete

<!-- REFLECTION_RUNNER: Describe one model from your project: table name, columns and types, primary key, and the request behind each CRUD operation. | id: database-models-project -->

---

## 7. References

- San Diego LabRats backend, `model/labrats_inquiry.py` and `api/labrats_inquiry_api.py`: [SDLabRats-Team/sdlabrats_backend](https://github.com/SDLabRats-Team/sdlabrats_backend)
- Open Coding Society Flask backend, `model/post.py` (`create` with rollback, the `_user_id` foreign key): [Open-Coding-Society/flask](https://github.com/Open-Coding-Society/flask)
- Python documentation, [`sqlite3`: DB-API 2.0 interface for SQLite databases](https://docs.python.org/3/library/sqlite3.html)
- SQLAlchemy documentation, [Models and Tables (Flask-SQLAlchemy)](https://flask-sqlalchemy.readthedocs.io/en/stable/models/)